# NEXT Honeycomb Cell — Acceptance Maps

2D histograms in the transverse plane (x0, y0) of the initial electron position.

**Part A — Electron acceptance:**
1. Electrons launched per bin
2. Electrons that entered the hole per bin
3. Fraction that entered the hole

**Part B — Photon fate (only electrons that entered the hole):**
4. Mean photons generated per electron
5. Mean photons reaching SiPM per electron
6. Mean photons escaped (top) per electron
7. Mean photons absorbed (wall) per electron

In [ ]:
import json
import numpy as np
import uproot
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import LogNorm

# Paths
BASE = "/Users/hernando/work/investigacion/NEXT/software/gala_sim"
JSON_FILE = f"{BASE}/el_electrons.json"
ROOT_FILE = f"{BASE}/build/output.root"

# Load electron data
with open(JSON_FILE) as f:
    data = json.load(f)

meta = data["metadata"]
electrons = data["electrons"]
hole_centres = np.array(meta["hole_centres"])
R_hole = meta["hole_radius_cm"]
pitch = meta["hole_pitch_cm"]
disk_r = meta["disk_radius_cm"]

# Load photon data
tree = uproot.open(ROOT_FILE)["photons"]
ph = tree.arrays(library="np")

print(f"Electrons: {len(electrons)}, Photons: {len(ph['electron_id'])}")
print(f"Hole radius: {R_hole} cm, Pitch: {pitch} cm, Disk radius: {disk_r} cm")

In [ ]:
# Build per-electron arrays
x0_all = np.array([e["x0"] for e in electrons])
y0_all = np.array([e["y0"] for e in electrons])
entered_all = np.array([e["entered_hole"] for e in electrons], dtype=bool)
total_photons_all = np.array([e["total_photons"] for e in electrons])

# Build per-electron photon fate from ROOT data (only for electrons that entered hole)
eid_set = set(ph["electron_id"])
photons_sipm = {}
photons_escaped = {}
photons_wall = {}
photons_total_root = {}

for eid in eid_set:
    m = ph["electron_id"] == eid
    photons_total_root[eid] = int(m.sum())
    photons_sipm[eid] = int(ph["reached_sipm"][m].sum())
    photons_escaped[eid] = int(ph["escaped_top"][m].sum())
    photons_wall[eid] = int(ph["absorbed_wall"][m].sum())

# Arrays for electrons that entered hole
x0_entered = np.array([e["x0"] for e in electrons if e["entered_hole"]])
y0_entered = np.array([e["y0"] for e in electrons if e["entered_hole"]])
eid_entered = np.array([e["electron_id"] for e in electrons if e["entered_hole"]])

ph_generated = np.array([total_photons_all[electrons.index(e)] for e in electrons if e["entered_hole"]])
# Use the electron_id indexing more directly
ph_generated = np.array([e["total_photons"] for e in electrons if e["entered_hole"]])
ph_sipm = np.array([photons_sipm.get(e["electron_id"], 0) for e in electrons if e["entered_hole"]])
ph_escaped = np.array([photons_escaped.get(e["electron_id"], 0) for e in electrons if e["entered_hole"]])
ph_wall = np.array([photons_wall.get(e["electron_id"], 0) for e in electrons if e["entered_hole"]])

print(f"Entered hole: {len(x0_entered)}/{len(electrons)}")
print(f"Mean photons generated: {ph_generated.mean():.1f}")
print(f"Mean photons → SiPM: {ph_sipm.mean():.1f}")
print(f"Mean photons escaped: {ph_escaped.mean():.1f}")
print(f"Mean photons wall abs: {ph_wall.mean():.1f}")

In [ ]:
# Helper: 2D binned mean
def binned_mean_2d(x, y, values, bins, range):
    """Compute mean of `values` in 2D bins of (x, y)."""
    sum_v, xedges, yedges = np.histogram2d(x, y, bins=bins, range=range, weights=values)
    count, _, _ = np.histogram2d(x, y, bins=bins, range=range)
    with np.warnings.catch_warnings():
        np.warnings.simplefilter("ignore", RuntimeWarning)
        mean_v = np.where(count > 0, sum_v / count, np.nan)
    return mean_v, xedges, yedges, count

# Helper: draw hole circles on an axis
def draw_holes(ax, centres, radius, color="white", lw=1, ls="-"):
    for hc in centres:
        ax.add_patch(Circle(hc, radius, fill=False, edgecolor=color, lw=lw, ls=ls))

# Bin setup
NBINS = 25
RANGE = [[-disk_r, disk_r], [-disk_r, disk_r]]

print(f"Bin size: {2*disk_r/NBINS*10:.2f} mm")

## Part A — Electron acceptance maps

Binned over **all** electrons (initial position in the generation disk).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# A1: Electrons launched per bin
h_launched, xedges, yedges = np.histogram2d(x0_all, y0_all, bins=NBINS, range=RANGE)
im1 = axes[0].imshow(h_launched.T, origin="lower", extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
                      cmap="cividis", aspect="equal")
draw_holes(axes[0], hole_centres, R_hole, color="red", lw=1.5)
axes[0].set_title("A1. Electrons launched")
axes[0].set_xlabel("x0 [cm]"); axes[0].set_ylabel("y0 [cm]")
fig.colorbar(im1, ax=axes[0], label="count")

# A2: Electrons that entered hole
h_entered, _, _ = np.histogram2d(x0_all[entered_all], y0_all[entered_all], bins=NBINS, range=RANGE)
im2 = axes[1].imshow(h_entered.T, origin="lower", extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
                      cmap="cividis", aspect="equal")
draw_holes(axes[1], hole_centres, R_hole, color="red", lw=1.5)
axes[1].set_title("A2. Electrons entering hole")
axes[1].set_xlabel("x0 [cm]"); axes[1].set_ylabel("y0 [cm]")
fig.colorbar(im2, ax=axes[1], label="count")

# A3: Fraction entering hole
with np.errstate(divide="ignore", invalid="ignore"):
    frac = np.where(h_launched > 0, h_entered / h_launched, np.nan)
im3 = axes[2].imshow(frac.T, origin="lower", extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
                      cmap="inferno", vmin=0, vmax=1, aspect="equal")
draw_holes(axes[2], hole_centres, R_hole, color="cyan", lw=1.5)
axes[2].set_title("A3. Fraction entering hole")
axes[2].set_xlabel("x0 [cm]"); axes[2].set_ylabel("y0 [cm]")
fig.colorbar(im3, ax=axes[2], label="fraction")

fig.suptitle(f"Part A — Electron acceptance  |  {meta['pressure_bar']} bar  |  "
             f"{len(electrons)} electrons  |  "
             f"Global acceptance: {meta['geometric_acceptance_pct']:.1f}%",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Part B — Photon fate maps (electrons that entered the hole only)

Each bin shows the **mean per electron** of photons generated, detected by SiPM, escaped top, and absorbed in wall. Only electrons that entered the hole contribute.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 12))

# All use colorblind-safe colormaps (viridis family)
datasets = [
    (ph_generated, "B1. Mean photons generated / e⁻", "plasma"),
    (ph_sipm,      "B2. Mean photons → SiPM / e⁻",   "viridis"),
    (ph_escaped,   "B3. Mean photons escaped (top) / e⁻", "inferno"),
    (ph_wall,      "B4. Mean photons absorbed (wall) / e⁻", "cividis"),
]

for ax, (vals, title, cmap) in zip(axes.flat, datasets):
    mean_map, xe, ye, count = binned_mean_2d(x0_entered, y0_entered, vals, NBINS, RANGE)
    mean_map_masked = np.where(count > 0, mean_map, np.nan)
    im = ax.imshow(mean_map_masked.T, origin="lower",
                   extent=[xe[0], xe[-1], ye[0], ye[-1]],
                   cmap=cmap, aspect="equal")
    draw_holes(ax, hole_centres, R_hole, color="white", lw=1.5)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("x0 [cm]"); ax.set_ylabel("y0 [cm]")
    fig.colorbar(im, ax=ax, label="mean photons / e⁻")

fig.suptitle(f"Part B — Photon fate (entered hole only)  |  {meta['pressure_bar']} bar  |  "
             f"{len(x0_entered)} electrons entered  |  "
             f"Mean SiPM acceptance: {ph_sipm.mean()/ph_generated.mean()*100:.1f}%",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
print("=" * 60)
print(f"  NEXT Honeycomb Cell — Summary ({meta['pressure_bar']} bar)")
print("=" * 60)
print(f"  Geometry: hole Ø{2*R_hole*10:.0f} mm, pitch {pitch*10:.0f} mm, height {meta['hole_height_cm']*10:.0f} mm")
print(f"  EL field: {meta['el_field_v_cm']:.0f} V/cm ({meta['el_reduced_field_kv_cm_bar']} kV/(cm·bar))")
print(f"  Drift field: {meta['drift_field_v_cm']:.0f} V/cm")
print(f"  Electrons: {len(electrons)} launched in disk r={disk_r*10:.0f} mm")
print("-" * 60)
print(f"  Part A — Electron acceptance:")
print(f"    Entered hole:    {entered_all.sum():4d} / {len(electrons)} = {entered_all.mean()*100:.1f}%")
print(f"    Hit wall:        {sum(e['hit_wall'] for e in electrons):4d}")
print(f"    Reached SiPM:    {sum(e['reached_sipm'] for e in electrons):4d}")
print("-" * 60)
print(f"  Part B — Photon fate (electrons in hole: {len(x0_entered)}):")
print(f"    Mean generated:  {ph_generated.mean():.1f} photons/e⁻")
print(f"    Mean → SiPM:     {ph_sipm.mean():.1f} photons/e⁻ ({ph_sipm.mean()/ph_generated.mean()*100:.1f}%)")
print(f"    Mean escaped:    {ph_escaped.mean():.1f} photons/e⁻ ({ph_escaped.mean()/ph_generated.mean()*100:.1f}%)")
print(f"    Mean wall abs:   {ph_wall.mean():.1f} photons/e⁻ ({ph_wall.mean()/ph_generated.mean()*100:.1f}%)")
print("=" * 60)